# GDELT Wildlife News Backfill
# Downloads GDELT GKG bulk files (one per day, 2015→today), filters for Indian
# wildlife articles, geocodes locations, saves news.json locally in Colab.
# Raw GDELT files are deleted immediately after parsing — peak disk use ~5 MB.
# At the end, Cell 8 downloads news.json straight to your PC.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q geopy

In [ ]:
# ── Cell 2: Paths (no Drive needed) ──────────────────────────────────────────
import os, io, json

NEWS_JSON_PATH = '/content/news.json'
print(f'Output will be saved to: {NEWS_JSON_PATH}')

In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────────
from datetime import datetime, timedelta, timezone

START_DATE = datetime(2015, 2, 18, tzinfo=timezone.utc)  # GDELT GKGv2 launch date
END_DATE   = datetime.now(timezone.utc)

# GDELT GKG themes that indicate wildlife/environment content
WILDLIFE_THEMES = {
    'ENV_POACHING', 'ENV_SPECIESCONSERVATION', 'ENV_DEFORESTATION',
    'ENV_FORESTRY', 'ENV_WILDLIFE', 'ENV_HABITAT', 'ENV_BIODIVERSITY',
    'TAX_FAUNAFISH', 'TAX_FAUNAMAMMALS', 'TAX_FAUNABIRDS', 'TAX_FAUNAREPTILES',
    'ENV_CLIMATECHANGE', 'NATURAL_DISASTER_DROUGHT', 'ENV_WATERWAYS',
}

# Fallback keyword check on source name / URL
WILDLIFE_KEYWORDS = [
    'tiger','leopard','elephant','rhino','lion','wolf','bear',
    'gharial','crocodile','python','vulture','bustard','dolphin',
    'wildlife','poaching','sanctuary','national park','forest reserve',
    'conservation','habitat','corridor','encroachment','species',
]

# India gazetteer — name → (lat, lon). Loaded from CSV in Cell 4.
INDIA_BOUNDS = dict(lat_min=6, lat_max=37, lon_min=68, lon_max=98)

print(f'Processing {START_DATE.date()} → {END_DATE.date()}')
total_days = (END_DATE - START_DATE).days
print(f'Total days: {total_days}')

In [ ]:
# ── Cell 4: Upload gazetteer + load it ────────────────────────────────────────
# Upload your data/india_pa_gazetteer.csv from the repo:
#   Left sidebar → Files icon → Upload → select india_pa_gazetteer.csv
# OR paste the path if you already put it in Drive.

import csv

GAZETTEER_PATH = '/content/india_pa_gazetteer.csv'  # adjust if in Drive

gazetteer = []  # list of {name, lat, lon}
try:
    with open(GAZETTEER_PATH, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            gazetteer.append({
                'name': row['name'].strip().lower(),
                'lat': float(row['lat']),
                'lon': float(row['lon']),
            })
    # Sort longest first for greedy matching
    gazetteer.sort(key=lambda x: len(x['name']), reverse=True)
    print(f'Loaded {len(gazetteer)} gazetteer entries')
except FileNotFoundError:
    print('WARNING: gazetteer not found — upload india_pa_gazetteer.csv')

In [ ]:
# ── Cell 5: Helper functions ───────────────────────────────────────────────────
import time
import json
import zipfile
import urllib.request
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

_geolocator = Nominatim(user_agent='wildlife-news-map-backfill')
_geocache = {}

def geocode(place_name):
    if place_name in _geocache:
        return _geocache[place_name]
    try:
        time.sleep(1)
        loc = _geolocator.geocode(f'{place_name}, India', timeout=10)
    except (GeocoderTimedOut, GeocoderServiceError):
        _geocache[place_name] = None
        return None
    if loc is None:
        _geocache[place_name] = None
        return None
    lat, lon = loc.latitude, loc.longitude
    b = INDIA_BOUNDS
    if not (b['lat_min'] <= lat <= b['lat_max'] and b['lon_min'] <= lon <= b['lon_max']):
        _geocache[place_name] = None
        return None
    result = (round(lat, 6), round(lon, 6))
    _geocache[place_name] = result
    return result


def extract_location_from_text(text):
    """Gazetteer-only extraction. Returns (name, lat, lon) or (None, None, None)."""
    lower = text.lower()
    for entry in gazetteer:
        if entry['name'] in lower:
            return entry['name'].title(), entry['lat'], entry['lon']
    return None, None, None


def extract_india_locations_from_gkg(v2locations_field):
    """
    Parse GDELT V2Locations field for India entries.
    Format: type#fullname#countrycode#adm1code#adm2code#lat#lon#featureid;...
    Returns list of (place_name, lat, lon) tuples.
    """
    results = []
    if not v2locations_field:
        return results
    for loc in v2locations_field.split(';'):
        parts = loc.split('#')
        if len(parts) < 7:
            continue
        country = parts[2]
        if country != 'IN':
            continue
        fullname = parts[1]
        try:
            lat = float(parts[5])
            lon = float(parts[6])
        except (ValueError, IndexError):
            continue
        b = INDIA_BOUNDS
        if b['lat_min'] <= lat <= b['lat_max'] and b['lon_min'] <= lon <= b['lon_max']:
            results.append((fullname, lat, lon))
    return results


def is_wildlife_relevant(themes_field, url, source_name):
    """True if article themes or URL suggest wildlife content."""
    themes = set(themes_field.upper().split(';')) if themes_field else set()
    if themes & WILDLIFE_THEMES:
        return True
    combined = (url + ' ' + source_name).lower()
    return any(kw in combined for kw in WILDLIFE_KEYWORDS)


def source_name_from_url(url):
    try:
        domain = url.split('/')[2].replace('www.', '')
        known = {
            'downtoearth.org': 'Down To Earth',
            'thewire.in': 'The Wire',
            'mongabay.com': 'Mongabay India',
            'thehindu.com': 'The Hindu',
            'timesofindia.indiatimes.com': 'Times of India',
            'hindustantimes.com': 'Hindustan Times',
            'ndtv.com': 'NDTV',
            'indianexpress.com': 'Indian Express',
            'scroll.in': 'Scroll',
            'firstpost.com': 'Firstpost',
            'deccanherald.com': 'Deccan Herald',
            'telegraphindia.com': 'The Telegraph',
            'business-standard.com': 'Business Standard',
            'thestatesman.com': 'The Statesman',
            'tribuneindia.com': 'The Tribune',
            'newindianexpress.com': 'New Indian Express',
            'dnaindia.com': 'DNA India',
            'outlookindia.com': 'Outlook India',
        }
        return known.get(domain, domain)
    except Exception:
        return 'Unknown'

print('Helper functions ready.')

In [ ]:
# ── Cell 6: Load existing news.json if present (resume support) ───────────────
if os.path.exists(NEWS_JSON_PATH):
    with open(NEWS_JSON_PATH, encoding='utf-8') as f:
        articles = json.load(f)
    print(f'Resuming: loaded {len(articles)} existing articles')
else:
    articles = []
    print('Starting fresh.')

seen_urls = {a['url'] for a in articles}

In [ ]:
# ── Cell 7: Main backfill loop ─────────────────────────────────────────────────
import urllib.request, zipfile, time
from datetime import datetime, timedelta, timezone

GDELT_BASE = 'http://data.gdeltproject.org/gdeltv2'
TMP_ZIP = '/content/gkg_tmp.zip'
TMP_CSV = '/content/gkg_tmp.csv'

current = START_DATE
total_days = (END_DATE - START_DATE).days
day_num = 0
new_count = 0

while current < END_DATE:
    day_num += 1
    date_str = current.strftime('%Y%m%d')
    url = f'{GDELT_BASE}/{date_str}120000.gkg.csv.zip'

    # Download
    try:
        urllib.request.urlretrieve(url, TMP_ZIP)
    except Exception as e:
        print(f'[{day_num}/{total_days}] {date_str} — download failed: {e}')
        current += timedelta(days=1)
        continue

    # Unzip
    try:
        with zipfile.ZipFile(TMP_ZIP) as z:
            csv_name = z.namelist()[0]
            z.extract(csv_name, '/content/')
            os.rename(f'/content/{csv_name}', TMP_CSV)
    except Exception as e:
        print(f'[{day_num}/{total_days}] {date_str} — unzip failed: {e}')
        if os.path.exists(TMP_ZIP): os.remove(TMP_ZIP)
        current += timedelta(days=1)
        continue

    # Parse
    day_articles = 0
    try:
        with open(TMP_CSV, encoding='utf-8', errors='replace') as f:
            for line in f:
                cols = line.rstrip('\n').split('\t')
                if len(cols) < 10:
                    continue
                date_raw   = cols[1] if len(cols) > 1 else ''
                source_raw = cols[3] if len(cols) > 3 else ''
                doc_url    = cols[4] if len(cols) > 4 else ''
                themes     = cols[7] if len(cols) > 7 else ''
                v2locs     = cols[9] if len(cols) > 9 else ''

                if not doc_url or doc_url in seen_urls:
                    continue
                india_locs = extract_india_locations_from_gkg(v2locs)
                if not india_locs:
                    continue
                if not is_wildlife_relevant(themes, doc_url, source_raw):
                    continue

                try:
                    pub_date = datetime.strptime(date_raw[:8], '%Y%m%d').strftime('%Y-%m-%d')
                except Exception:
                    pub_date = f'{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}'

                india_locs.sort(key=lambda x: len(x[0]), reverse=True)
                place_name, lat, lon = india_locs[0]

                gz_name, gz_lat, gz_lon = extract_location_from_text(place_name)
                if gz_name:
                    place_name, lat, lon = gz_name, gz_lat, gz_lon

                headline = source_raw or doc_url.split('/')[-1].replace('-', ' ').replace('_', ' ')

                articles.append({
                    'headline': headline,
                    'url': doc_url,
                    'source': source_name_from_url(doc_url),
                    'published': pub_date,
                    'place_name': place_name,
                    'lat': round(lat, 6),
                    'lon': round(lon, 6),
                })
                seen_urls.add(doc_url)
                day_articles += 1
                new_count += 1

    except Exception as e:
        print(f'[{day_num}/{total_days}] {date_str} — parse error: {e}')

    # Delete raw files immediately
    for f in [TMP_ZIP, TMP_CSV]:
        if os.path.exists(f): os.remove(f)

    if day_num % 30 == 0 or day_articles > 0:
        print(f'[{day_num}/{total_days}] {date_str} — {day_articles} new (total: {new_count})')

    # Checkpoint every 30 days
    if day_num % 30 == 0:
        articles.sort(key=lambda a: a.get('published', ''), reverse=True)
        with open(NEWS_JSON_PATH, 'w', encoding='utf-8') as f:
            json.dump(articles, f, indent=2, ensure_ascii=False)
        print(f'  → Checkpoint saved ({len(articles)} articles)')

    current += timedelta(days=1)

print(f'\nDone! {new_count} new articles found.')

In [ ]:
# ── Cell 8: Final save + download to your PC ─────────────────────────────────
articles.sort(key=lambda a: a.get('published', ''), reverse=True)

with open(NEWS_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(articles, f, indent=2, ensure_ascii=False)

print(f'Total articles: {len(articles)}')
print(f'Date range: {articles[-1]["published"]} → {articles[0]["published"]}')
for a in articles[:5]:
    print(f'  [{a["published"]}] {a["place_name"]} | {a["source"]}')

# Download news.json straight to your PC
from google.colab import files
files.download(NEWS_JSON_PATH)